In [3]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score
from copy import deepcopy
from torch_geometric.transforms import NormalizeFeatures

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 1. GCN MODEL
# ============================================================
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers, dropout_p):
        super().__init__()
        self.num_layers = num_layers
        self.dropout_p = dropout_p
        self.convs = nn.ModuleList()
        
        if num_layers == 0:
            self.classifier = nn.Linear(in_dim, out_dim)
        else:
            self.convs.append(GCNConv(in_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_dim, hidden_dim))
            if num_layers > 1:
                self.convs.append(GCNConv(hidden_dim, out_dim))
            else:
                self.convs[0] = GCNConv(in_dim, out_dim)

    def forward(self, x, edge_index):
        if self.num_layers == 0:
            out = self.classifier(x)
            return F.log_softmax(out, dim=1), x
        
        h = x
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < len(self.convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout_p, training=self.training)
        return F.log_softmax(h, dim=1), h

# ============================================================
#  ECE
# ============================================================
def compute_ece(probs, y_true, n_bins=20):
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(y_true)
    bins = torch.linspace(0, 1, n_bins+1, device=probs.device)
    ece = 0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        count = mask.sum().item()
        if count > 0:
            acc = accuracies[mask].float().mean().item()
            conf = confidences[mask].mean().item()
            ece += (count / len(probs)) * abs(acc - conf)
    return ece

# ============================================================

def run_best_hyper_experiment():
    layers_list = [3]
    hidden_dims = [64]
    dropouts = [0.5]
    norms = [True]
    seeds = [0, 42, 7]
    
    all_configs_results = []

    dataset_no_norm = Planetoid(root='./data', name='Cora')
    dataset_with_norm = Planetoid(root='./data', name='Cora', transform=NormalizeFeatures())

    for use_norm in norms:
        data = (dataset_with_norm[0] if use_norm else dataset_no_norm[0]).to(device)
        norm_str = "WITH Norm" if use_norm else "WITHOUT Norm"
        
        for n_layers in layers_list:
            h_space = hidden_dims if n_layers > 0 else [None]
            d_space = dropouts if n_layers > 0 else [None]
            
            for h_dim in h_space:
                for d_p in d_space:
                    seed_accs, seed_eces, seed_val_losses = [], [], []

                    for s in seeds:
                        torch.manual_seed(s)
                        model = GCN(data.num_features, h_dim or 16, dataset_no_norm.num_classes, n_layers, d_p or 0).to(device)
                        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
                        
                        best_val_loss = float('inf')
                        patience, counter = 50, 0
                        best_state = None

                        for epoch in range(200):
                            model.train()
                            optimizer.zero_grad()
                            out, _ = model(data.x, data.edge_index)
                            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
                            loss.backward()
                            optimizer.step()
                            
                            model.eval()
                            with torch.no_grad():
                                val_out, _ = model(data.x, data.edge_index)
                                val_loss = F.nll_loss(val_out[data.val_mask], data.y[data.val_mask])
                            
                            if val_loss < best_val_loss:
                                best_val_loss = val_loss
                                best_state = deepcopy(model.state_dict())
                                counter = 0
                            else:
                                counter += 1
                            if counter >= patience: break

                        model.load_state_dict(best_state)
                        model.eval()
                        with torch.no_grad():
                            logits, _ = model(data.x, data.edge_index)
                            # probs = F.softmax(logits, dim=1)[data.test_mask]
                            probs = logits[data.test_mask]
                            y_test = data.y[data.test_mask]
                            acc = accuracy_score(y_test.cpu(), probs.argmax(dim=1).cpu())
                            ece = compute_ece(probs, y_test)
                            
                            seed_accs.append(acc)
                            seed_eces.append(ece)
                            seed_val_losses.append(best_val_loss.item())

                    all_configs_results.append({
                        "Layers": n_layers,
                        "Norm": norm_str,
                        "Hidden": h_dim,
                        "Dropout": d_p,
                        "Val_Loss_Mean": np.mean(seed_val_losses),
                        "Acc_Mean": np.mean(seed_accs),
                        "Acc_Std": np.std(seed_accs),
                        "ECE_Mean": np.mean(seed_eces),
                        "ECE_Std": np.std(seed_eces)
                    })

    # Selezione del migliore per ogni (Layers, Norm) basata su Val_Loss_Mean
    df_all = pd.DataFrame(all_configs_results)
    best_results = df_all.loc[df_all.groupby(['Layers', 'Norm'])['Val_Loss_Mean'].idxmin()]
    return best_results

# ============================================================
# Stampa della Tabella Formattata
# ============================================================
df_best = run_best_hyper_experiment()

def print_pretty_best_table(dataframe):
    df_sorted = dataframe.sort_values(by='Layers', kind='stable')

    print("\n" + "="*125)
    print(f"{'RESULTS OF BEST CONFIGURATION PER LAYER':^125}")
    print("="*125)
    
    header = f"{'Layers':<8} | {'Norm':<14} | {'Best Config (H, D)':<20} | {'Accuracy':<18} | {'ECE (15 bins)':<18}"
    print(header)
    print("-" * len(header))
    
    last_layer = None
    for _, row in df_sorted.iterrows():
        if last_layer is not None and row['Layers'] != last_layer:
            print("-" * len(header))
            
        config_str = f"H:{row['Hidden']}, D:{row['Dropout']}" if row['Layers'] > 0 else "N/A (Linear)"
        acc_str = f"{row['Acc_Mean']:.4f} ± {row['Acc_Std']:.4f}"
        ece_str = f"{row['ECE_Mean']:.4f} ± {row['ECE_Std']:.4f}"
        
        line = (f"{row['Layers']:<8} | {row['Norm']:<14} | {config_str:<20} | {acc_str:<18} | {ece_str:<18}")
        print(line)
        last_layer = row['Layers']
        
    print("="*125)

print_pretty_best_table(df_best)


                                           RESULTS OF BEST CONFIGURATION PER LAYER                                           
Layers   | Norm           | Best Config (H, D)   | Accuracy           | ECE (15 bins)     
------------------------------------------------------------------------------------------
3        | WITH Norm      | H:64, D:0.5          | 0.8043 ± 0.0012    | 0.0000 ± 0.0000   


## LCE

In [22]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import NormalizeFeatures

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def compute_lce_with_ess(probs, y_true, embeddings, n_bins=15, gamma=0.2):
    """
    Implementazione della LCE basata sull'Eq. 4 di Luo et al. (2022).
    """
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(y_true).float()
    
    # Il paper suggerisce di usare feature normalizzate o ridotte (PCA/t-SNE)
    # Qui usiamo gli embeddings normalizzati per coerenza con il kernel RBF/Laplaciano
    emb_norm = F.normalize(embeddings, p=2, dim=1)
    
    # Calcolo del Kernel (Laplaciano come nel paper, Eq. a pag 4)
    # k(x, x') = exp(- ||phi(x) - phi(x')||_1 / (d * gamma))
    dist_l1 = torch.cdist(emb_norm, emb_norm, p=1)
    d = embeddings.shape[1] # dimensione feature
    kernel_matrix = torch.exp(-dist_l1 / (d * gamma))
    
    bins = torch.linspace(0, 1, n_bins + 1)
    lce_per_point = torch.zeros(len(probs)).to(probs.device)

    for i in range(n_bins):
        # b(x): indici dei punti nello stesso bin di confidenza
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if i == n_bins - 1: # Include l'estremo 1.0 nell'ultimo bin
            mask = (confidences >= bins[i]) & (confidences <= bins[i+1])
            
        indices = torch.where(mask)[0]
        
        if len(indices) > 0:
            # Calcolo (p_hat - accuracy) per i punti nel bin
            diffs = confidences[indices] - accuracies[indices]
            
            # Sottomatrice del kernel per il bin corrente
            k_bin = kernel_matrix[indices][:, indices]
            
            # Numeratore: somma pesata delle differenze (Eq. 4)
            numerator = torch.mv(k_bin, diffs)
            
            # Denominatore: somma dei pesi del kernel
            denominator = k_bin.sum(dim=1)
            
            # LCE per ogni punto nel bin
            lce_per_point[indices] = torch.abs(numerator / (denominator + 1e-9))

    # Il paper definisce la LCE media sul dataset o la MLCE (il massimo)
    avg_lce = lce_per_point.mean().item()
    
    # Calcolo ESS (Effective Sample Size) per monitorare la densità locale
    # ESS = (sum K)^2 / sum(K^2)
    ess_list = []
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        indices = torch.where(mask)[0]
        if len(indices) > 0:
            k_bin = kernel_matrix[indices][:, indices]
            ess = (k_bin.sum(dim=1)**2) / ((k_bin**2).sum(dim=1) + 1e-9)
            ess_list.append(ess.mean().item())
            
    return avg_lce, np.mean(ess_list) if ess_list else 0

# ============================================================
# model
# ============================================================
class FlexibleGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        
        if num_layers == 0:
            self.classifier = nn.Linear(in_dim, out_dim)
        else:
            self.convs.append(GCNConv(in_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_dim, hidden_dim))
            if num_layers > 1:
                self.convs.append(GCNConv(hidden_dim, out_dim))
            else:
                self.convs[0] = GCNConv(in_dim, out_dim)

    def forward(self, x, edge_index):
        if self.num_layers == 0:
            out = self.classifier(x)
            return F.log_softmax(out, dim=1), x
        
        h = x
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < len(self.convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=0.5, training=self.training)
        return F.log_softmax(h, dim=1), h

def run_ess_analysis(use_normalization=False):
    label = "CON NORM" if use_normalization else "SENZA NORM"
    print(f"\n{'='*90}\nResults: {label}\n{'='*90}")

    transform = NormalizeFeatures() if use_normalization else None
    dataset = Planetoid(root='./data', name='Cora', transform=transform)
    data = dataset[0].to(device)

    manual_seeds = [0, 42, 123]  
    bins_list = [10, 15, 20]
    gamma_list = [0.1, 0.5, 0.7, 1.0, 2.0, 5.0, 10]
    test_mask = data.test_mask

    all_lce_results = { (nb, g): [] for nb in bins_list for g in gamma_list }
    all_ess_results = { (nb, g): [] for nb in bins_list for g in gamma_list }

    for s in manual_seeds:
        torch.manual_seed(s)
        np.random.seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(s)

        model = FlexibleGCN(data.num_features, 64, dataset.num_classes, num_layers=3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
        
        for _ in range(500):
            model.train()
            optimizer.zero_grad()
            out, _ = model(data.x, data.edge_index)
            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            logits, embs = model(data.x, data.edge_index)
            probs = F.softmax(logits, dim=1)
        
        for nb in bins_list:
            for g in gamma_list:
                lce_v, ess_v = compute_lce_with_ess(
                    probs[test_mask], data.y[test_mask], embs[test_mask], n_bins=nb, gamma=g
                )
                all_lce_results[(nb, g)].append(lce_v)
                all_ess_results[(nb, g)].append(ess_v)

    lce_final_table = []
    ess_final_table = []

    for nb in bins_list:
        l_row, e_row = {"Bins": nb}, {"Bins": nb}
        for g in gamma_list:

            lce_vals = all_lce_results[(nb, g)]
            ess_vals = all_ess_results[(nb, g)]
            
            l_row[f"G={g}"] = f"{np.mean(lce_vals):.4f} ± {np.std(lce_vals):.4f}"
            e_row[f"G={g}"] = f"{np.mean(ess_vals):.2f} ± {np.std(ess_vals):.2f}"
            
        lce_final_table.append(l_row)
        ess_final_table.append(e_row)

    print("\n--- TABELLA LCE (Media ± Std ) ---")
    print(pd.DataFrame(lce_final_table).to_string(index=False))
    print("\n--- TABELLA ESS MEDIA (Media ± Std su 3 Seed) ---")
    print(pd.DataFrame(ess_final_table).to_string(index=False))

# Esecuzione
run_ess_analysis(use_normalization=True)


Results: CON NORM

--- TABELLA LCE (Media ± Std ) ---
 Bins           G=0.1           G=0.5           G=0.7           G=1.0           G=2.0           G=5.0            G=10
   10 0.0886 ± 0.0081 0.0625 ± 0.0097 0.0626 ± 0.0091 0.0627 ± 0.0087 0.0629 ± 0.0084 0.0632 ± 0.0084 0.0633 ± 0.0084
   15 0.1022 ± 0.0074 0.0659 ± 0.0085 0.0659 ± 0.0084 0.0660 ± 0.0082 0.0662 ± 0.0079 0.0663 ± 0.0078 0.0663 ± 0.0077
   20 0.1115 ± 0.0048 0.0663 ± 0.0076 0.0657 ± 0.0074 0.0656 ± 0.0073 0.0656 ± 0.0074 0.0657 ± 0.0075 0.0658 ± 0.0075

--- TABELLA ESS MEDIA (Media ± Std su 3 Seed) ---
 Bins        G=0.1         G=0.5         G=0.7         G=1.0         G=2.0         G=5.0          G=10
   10 24.28 ± 0.78 113.38 ± 1.02 119.18 ± 0.54 122.23 ± 0.27 124.34 ± 0.07 124.90 ± 0.01 124.97 ± 0.00
   15 15.96 ± 0.36  75.32 ± 0.70  79.32 ± 0.37  81.42 ± 0.18  82.88 ± 0.05  83.26 ± 0.01  83.32 ± 0.00
   20 11.95 ± 0.24  56.34 ± 0.51  59.41 ± 0.27  61.03 ± 0.14  62.15 ± 0.03  62.45 ± 0.01  62.49 ± 0.00
